<a href="https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/03_training_colab_mlflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroScan AI: Entrenamiento Acelerado en Google Colab Pro con MLflow
## Módulo 3: Databricks e IA Aplicada - Proyecto Final

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduardCY/m3-proyecto-final-databricks-ia/blob/main/notebooks/03_training_colab_mlflow.ipynb)

Este notebook implementa la **Etapa 3**:
1. Detección y activación de aceleración por GPU (NVIDIA A100 / V100 / L4) y **Mixed Precision (FP16)**.
2. Sincronización automática con Google Drive para respaldar checkpoints y base de datos SQLite de MLflow.
3. Ejecución de **5 Experimentos de Transfer Learning**:
   - **Run 1:** ResNet50 Base Congelada (LR 1e-3, Dropout 0.3)
   - **Run 2:** ResNet50 Base Congelada (LR 1e-4, Dropout 0.5)
   - **Run 3:** ResNet50 Fine-Tuning capa 143 (LR 1e-5, Dropout 0.3)
   - **Run 4 (Variante 1):** ResNet50 Fine-Tuning profundo capa 120 + Cosine Decay
   - **Run 5 (Variante 2):** EfficientNetB2 Transfer Learning (Comparación Multiarquitectura)


In [1]:
# 1. Configuración de Entorno y Detección de GPU
import os
import sys
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Aceleradora Detectada: {gpus}")
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("🚀 Mixed Precision FP16 habilitada para Tensor Cores.")
else:
    print("⚠️ No se detectó GPU. Ejecutando en CPU.")


TensorFlow: 2.20.0
✅ GPU Aceleradora Detectada: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
🚀 Mixed Precision FP16 habilitada para Tensor Cores.


In [2]:
# 2. Montar Google Drive para Respaldo Resiliente
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/m3_project'
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/backups", exist_ok=True)
    print(f"✅ Google Drive conectado en: {DRIVE_DIR}")
except Exception as e:
    print(f"Ejecutando en entorno local sin montaje de Colab Drive: {e}")


Ejecutando en entorno local sin montaje de Colab Drive: mount failed


In [3]:
# 3. Importar Módulos del Sistema
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import BATCH_SIZE, CLASSES, SPLIT_MANIFEST_PATH
from src.preprocessing import create_tf_dataset
from src.train import train_and_log_run

df_split = pd.read_csv(SPLIT_MANIFEST_PATH)
train_ds = create_tf_dataset(df_split, split_name="train", batch_size=BATCH_SIZE, model_type="resnet50")
val_ds = create_tf_dataset(df_split, split_name="val", batch_size=BATCH_SIZE, model_type="resnet50")
test_ds = create_tf_dataset(df_split, split_name="test", batch_size=BATCH_SIZE, model_type="resnet50")
print(f"Datasets listos con {len(df_split)} imágenes en total.")


ModuleNotFoundError: No module named 'src'

### 4. Ejecución de los 5 Experimentos en MLflow


In [ ]:
# Run 1: ResNet50 Frozen Base (LR 1e-3, Dropout 0.3)
m1, h1, met1 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_1_ResNet50_Frozen_1e3",
    backbone_name="resnet50",
    epochs=12, learning_rate=1e-3, dropout=0.3,
    base_trainable=False
)


In [ ]:
# Run 2: ResNet50 Frozen Base (LR 1e-4, Dropout 0.5)
m2, h2, met2 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_2_ResNet50_Frozen_1e4",
    backbone_name="resnet50",
    epochs=12, learning_rate=1e-4, dropout=0.5,
    base_trainable=False
)


In [ ]:
# Run 3: ResNet50 Fine-Tuning capa 143 (LR 1e-5, Dropout 0.3)
m3, h3, met3 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_3_ResNet50_FineTune_143",
    backbone_name="resnet50",
    epochs=15, learning_rate=1e-5, dropout=0.3,
    base_trainable=True, fine_tune_at=143
)


In [ ]:
# Run 4 (Variante 1): ResNet50 Fine-Tuning profundo capa 120 + Cosine Decay
m4, h4, met4 = train_and_log_run(
    train_dataset=train_ds, val_dataset=val_ds, test_dataset=test_ds,
    run_name="Run_4_ResNet50_DeepFineTune_120_Cosine",
    backbone_name="resnet50",
    epochs=15, learning_rate=3e-5, dropout=0.3,
    base_trainable=True, fine_tune_at=120,
    use_cosine_decay=True
)


In [ ]:
# Run 5 (Variante 2): EfficientNetB2 Transfer Learning (Comparación Multiarquitectura)
train_eff = create_tf_dataset(df_split, split_name="train", batch_size=BATCH_SIZE, model_type="efficientnetb2")
val_eff = create_tf_dataset(df_split, split_name="val", batch_size=BATCH_SIZE, model_type="efficientnetb2")
test_eff = create_tf_dataset(df_split, split_name="test", batch_size=BATCH_SIZE, model_type="efficientnetb2")

m5, h5, met5 = train_and_log_run(
    train_dataset=train_eff, val_dataset=val_eff, test_dataset=test_eff,
    run_name="Run_5_EfficientNetB2_Transfer",
    backbone_name="efficientnetb2",
    epochs=12, learning_rate=1e-3, dropout=0.3,
    base_trainable=False
)


### 5. Sincronización Final de Resultados a Google Drive


In [ ]:
from scripts.sync_drive import backup_to_gdrive
backup_to_gdrive()
print("✅ Todos los checkpoints, modelos y la base de datos de MLflow fueron respaldados en Google Drive.")
